# CHSA Medical Triage Agent - Kaggle Serving

Kaggle/GPU notebook for serving the CHSA POC with vLLM plus the FastAPI triage wrapper. It mirrors the Colab serving notebook but uses Kaggle paths and Kaggle Secrets. Keep the serving cell running while you test the API.


## Runtime checks

Enable a GPU accelerator in Kaggle before model serving. A T4/P100-class GPU is expected for this POC path.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print("python=", sys.version)
print("cwd=", Path.cwd())
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found; enable a Kaggle GPU accelerator before model serving.")

## Clone or update repository

On Kaggle the checkout lives under `/kaggle/working`. If you run this notebook locally from VS Code, it reuses the current repository when possible.


In [ ]:
REPO_URL = "https://github.com/Nhkp/medical-triage-agent.git"
if Path("/kaggle/working").exists():
    DEFAULT_REPO_DIR = Path("/kaggle/working/medical-triage-agent")
else:
    DEFAULT_REPO_DIR = Path.cwd() / ".kaggle_runtime" / "medical-triage-agent"

REPO_DIR = DEFAULT_REPO_DIR

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif Path.cwd().name == "medical-triage-agent" and (Path.cwd() / ".git").exists():
    REPO_DIR = Path.cwd()
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout; remove it or choose another path"
        )
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("repo=", Path.cwd())

## Install serving dependencies

Kaggle owns the CUDA runtime, so install serving packages into the notebook environment instead of creating a project `.venv`.


In [ ]:
!python -m pip install -q -U fastapi uvicorn vllm pyngrok huggingface_hub

## Load Hugging Face and ngrok credentials

Priority order: `.env`, environment variables, Kaggle Secrets, then interactive prompt. Secrets stay in the process environment only.


In [ ]:
from getpass import getpass


def load_dotenv(path: Path) -> dict[str, str]:
    values = {}
    if not path.exists():
        return values
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip(chr(34)).strip(chr(39))
    return values


def kaggle_secret(name: str) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except (ImportError, KeyError, RuntimeError, OSError):
        return None


dotenv = load_dotenv(Path(".env"))

token = dotenv.get("HF_TOKEN") or os.environ.get("HF_TOKEN") or kaggle_secret("HF_TOKEN")
if not token:
    token = getpass("HF_TOKEN: ")
os.environ["HF_TOKEN"] = token

ngrok_token = (
    dotenv.get("NGROK_AUTHTOKEN")
    or os.environ.get("NGROK_AUTHTOKEN")
    or kaggle_secret("NGROK_AUTHTOKEN")
)
if ngrok_token:
    os.environ["NGROK_AUTHTOKEN"] = ngrok_token

for key in ("HF_DPO_MODEL_REPO", "HF_SFT_MODEL_REPO", "VLLM_MODEL_ID"):
    if dotenv.get(key):
        os.environ[key] = dotenv[key]

print("HF token loaded:", bool(os.environ.get("HF_TOKEN")))
print("ngrok token loaded:", bool(os.environ.get("NGROK_AUTHTOKEN")))

## Select model repo

Prefer the DPO adapter repo, then SFT adapter repo, then the base Qwen model for smoke serving.


In [ ]:
BASE_MODEL_REPO = os.environ.get("VLLM_BASE_MODEL_ID") or "Qwen/Qwen3-1.7B-Base"
ADAPTER_REPO = (
    os.environ.get("HF_DPO_MODEL_REPO")
    or os.environ.get("HF_SFT_MODEL_REPO")
    or os.environ.get("VLLM_LORA_ADAPTER")
    or "Lokhidor/medical-triage-qwen3-dpo-lora"
)
LORA_NAME = os.environ.get("VLLM_MODEL_ID") or "medical-triage-dpo"
os.environ["VLLM_BASE_MODEL_ID"] = BASE_MODEL_REPO
os.environ["VLLM_LORA_ADAPTER"] = ADAPTER_REPO
os.environ["VLLM_MODEL_ID"] = LORA_NAME
print("BASE_MODEL_REPO=", BASE_MODEL_REPO)
print("ADAPTER_REPO=", ADAPTER_REPO)
print("LORA_NAME=", LORA_NAME)

## Dry-run serving commands

This does not load the model. It prints the exact vLLM and FastAPI commands the script will launch.


In [ ]:
!python scripts/serve_colab.py --base-model "$BASE_MODEL_REPO" --adapter "$ADAPTER_REPO" --lora-name "$LORA_NAME" --dry-run

## Launch vLLM + FastAPI + ngrok

Run this cell and keep it running. Stop it with the notebook interrupt button when finished. If you do not need an external URL, remove `--ngrok` and test with `127.0.0.1` from another cell.


In [ ]:
!python scripts/serve_colab.py --base-model "$BASE_MODEL_REPO" --adapter "$ADAPTER_REPO" --lora-name "$LORA_NAME" --ngrok

## Local endpoint tests

Use these from another cell while the serving cell is running.


In [ ]:
!curl -s http://127.0.0.1:8080/health

In [ ]:
!curl -s -X POST http://127.0.0.1:8080/triage \
  -H "Content-Type: application/json" \
  -d '{"symptoms":["douleur thoracique","difficulte respiratoire"]}'

## Audit lookup

Paste an `audit_id` returned by `/triage`. Audit output must remain metadata-only.


In [ ]:
AUDIT_ID = "audit_REPLACE_ME"
!curl -s http://127.0.0.1:8080/audit/$AUDIT_ID

## Robustness and latency checks

Run these after the API is up. They write outputs under `outputs/evaluations`, which is ignored by git.


In [ ]:
!python scripts/evaluate_robustness.py --url http://127.0.0.1:8080
!python scripts/evaluate_latency.py --url http://127.0.0.1:8080 --iterations 12

## Persist evaluation outputs

Kaggle working storage can disappear after the session. Zip the non-sensitive evaluation outputs before downloading or saving them as notebook output.


In [ ]:
!zip -r outputs-evaluations.zip outputs/evaluations || true

## Debug: processes and ports

Use these cells if a previous run was interrupted or a port is already busy.


In [ ]:
!ps -ef | grep -E 'vllm|uvicorn|serve_colab' | grep -v grep || true

In [ ]:
import socket

for port in (8000, 8080):
    sock = socket.socket()
    result = sock.connect_ex(("127.0.0.1", port))
    print(port, "open" if result == 0 else "closed")
    sock.close()

In [ ]:
# Use only if you need to clean up stuck serving processes.
# !pkill -f 'vllm.entrypoints.openai.api_server' || true
# !pkill -f 'uvicorn medical_triage_agent.api:create_app' || true
# !pkill -f 'scripts/serve_colab.py' || true
